# Siamese Hybrid Network - ConvNeXt x Swin Transformer
## Signature Verification via Contrastive Learning

This notebook implements a **production-grade Siamese architecture** that fuses a
**ConvNeXt** backbone (local textural features) with a **Swin Transformer**
backbone (global structural context) through a **cross-attention fusion module**.

The model is trained with **contrastive loss** to map the structural DNA of
handwriting into a compact 256-d embedding space, providing robust defence
against skilled forgeries.

### Pipeline
| # | Section |
|---|---------|
| 1 | Setup & Imports |
| 2 | Drive Mount & Config |
| 3 | Hyperparameters |
| 4 | CEDAR Dataset |
| 5 | Augmentation |
| 6 | DataLoaders |
| 7 | Hybrid Backbone |
| 8 | Contrastive Loss |
| 9 | Training |
| 10 | Evaluation |
| 11 | Visualisation |
| 12 | Inference |
| 13 | Export |


## 1. Environment Setup


In [ ]:
!pip install -q timm scikit-learn matplotlib seaborn tqdm


In [ ]:
import os, re, glob, random, warnings, time, copy, math
from collections import defaultdict
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import timm

from sklearn.metrics import (roc_curve, auc,
                              confusion_matrix, classification_report)
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')


## 2. Mount Google Drive & Configure Paths


In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not running in Colab - skipping Drive mount.')

# >>>>>>>  EDIT THESE PATHS  <<<<<<<
if IN_COLAB:
    DATA_DIR = '/content/drive/MyDrive/signatures'   # folder with full_org/ & full_forg/
else:
    DATA_DIR = 'signatures'

CHECKPOINT_DIR = 'checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'Data directory  : {DATA_DIR}')
print(f'Checkpoint dir  : {CHECKPOINT_DIR}')


## 3. Hyperparameters


In [ ]:
IMG_SIZE            = 224
EMBED_DIM           = 256
BATCH_SIZE          = 16
NUM_EPOCHS          = 25
LEARNING_RATE       = 3e-4
WEIGHT_DECAY        = 1e-4
CONTRASTIVE_MARGIN  = 2.0
TRAIN_SPLIT         = 0.8
PAIRS_PER_WRITER    = 60
NUM_WORKERS         = 2
SEED                = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
print('Hyperparameters set.')


## 4. Dataset - CEDAR-Aware Pairing

CEDAR naming convention:
- Genuine : `original_{writerID}_{sampleID}.png` (55 writers x 24 samples)
- Forged  : `forgeries_{writerID}_{sampleID}.png` (55 writers x 24 samples)

| Pair Type | Label | How Built |
|-----------|-------|-----------|
| Genuine | 1 | Two different originals of the same writer |
| Forged  | 0 | One original + one forgery of the same writer |


In [ ]:
class CEDARSignatureDataset(Dataset):
    '''CEDAR-aware Siamese pair dataset with balanced genuine/forged pairs.'''

    def __init__(self, data_dir, writer_ids, transform=None,
                 pairs_per_writer=60):
        super().__init__()
        self.transform = transform

        org_dir  = os.path.join(data_dir, 'full_org')
        forg_dir = os.path.join(data_dir, 'full_forg')

        # Group files by writer ID
        org_by_writer  = defaultdict(list)
        forg_by_writer = defaultdict(list)

        for f in sorted(glob.glob(os.path.join(org_dir, '*.png'))):
            wid = self._parse_writer(os.path.basename(f), 'original')
            if wid in writer_ids:
                org_by_writer[wid].append(f)

        for f in sorted(glob.glob(os.path.join(forg_dir, '*.png'))):
            wid = self._parse_writer(os.path.basename(f), 'forgeries')
            if wid in writer_ids:
                forg_by_writer[wid].append(f)

        # Build balanced pairs
        self.pairs = []
        for wid in writer_ids:
            orgs  = org_by_writer.get(wid, [])
            forgs = forg_by_writer.get(wid, [])
            if len(orgs) < 2:
                continue

            # Genuine pairs (label = 1)
            gen_combos = list(combinations(orgs, 2))
            random.shuffle(gen_combos)
            for a, b in gen_combos[:pairs_per_writer]:
                self.pairs.append((a, b, 1))

            # Forged pairs (label = 0)
            neg_pairs = [(o, fg) for o in orgs for fg in forgs]
            random.shuffle(neg_pairs)
            for o, fg in neg_pairs[:pairs_per_writer]:
                self.pairs.append((o, fg, 0))

        random.shuffle(self.pairs)
        print(f'  Writer IDs: {sorted(writer_ids)[:10]}...  |  '
              f'Total pairs: {len(self.pairs)}')

    @staticmethod
    def _parse_writer(filename, prefix):
        '''original_10_3.png -> 10'''
        m = re.match(rf'{prefix}_(\d+)_\d+\.png', filename)
        return int(m.group(1)) if m else -1

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path1, path2, label = self.pairs[idx]
        img1 = Image.open(path1).convert('RGB')
        img2 = Image.open(path2).convert('RGB')
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        return img1, img2, torch.tensor(label, dtype=torch.float32)


## 5. Data Augmentation

Signature-specific transforms that simulate natural pen variation
without destroying structural identity.


In [ ]:
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.RandomPerspective(distortion_scale=0.15, p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])
print('Transforms defined.')


## 6. Build Datasets & DataLoaders


In [ ]:
# Discover all writer IDs
all_org_files = glob.glob(os.path.join(DATA_DIR, 'full_org', '*.png'))
writer_ids = sorted({
    int(re.match(r'original_(\d+)_\d+\.png', os.path.basename(f)).group(1))
    for f in all_org_files
    if re.match(r'original_(\d+)_\d+\.png', os.path.basename(f))
})
print(f'Found {len(writer_ids)} writers')

# Split by writer
split_idx     = int(len(writer_ids) * TRAIN_SPLIT)
train_writers = writer_ids[:split_idx]
test_writers  = writer_ids[split_idx:]
print(f'Train writers: {len(train_writers)}  |  Test writers: {len(test_writers)}')

print('\nBuilding train dataset...')
train_dataset = CEDARSignatureDataset(
    DATA_DIR, train_writers, transform=train_transform,
    pairs_per_writer=PAIRS_PER_WRITER)

print('Building test dataset...')
test_dataset = CEDARSignatureDataset(
    DATA_DIR, test_writers, transform=val_transform,
    pairs_per_writer=PAIRS_PER_WRITER)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=True)
print(f'\nTrain batches: {len(train_loader)}  |  Test batches: {len(test_loader)}')


### 6.1 Visualise Sample Pairs


In [ ]:
def show_pairs(dataset, n=4):
    inv = T.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225])
    fig, axes = plt.subplots(n, 2, figsize=(8, 3*n))
    for i in range(n):
        idx = random.randint(0, len(dataset)-1)
        img1, img2, label = dataset[idx]
        img1 = inv(img1).clamp(0,1).permute(1,2,0).numpy()
        img2 = inv(img2).clamp(0,1).permute(1,2,0).numpy()
        tag = 'GENUINE' if label.item()==1 else 'FORGED'
        color = '#2ecc71' if label.item()==1 else '#e74c3c'
        axes[i,0].imshow(img1); axes[i,0].set_title(f'A [{tag}]', color=color)
        axes[i,1].imshow(img2); axes[i,1].set_title(f'B [{tag}]', color=color)
        axes[i,0].axis('off'); axes[i,1].axis('off')
    plt.suptitle('Sample Signature Pairs', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_pairs(train_dataset, n=5)


## 7. Model Architecture - Hybrid Backbone

| Component | Role |
|-----------|------|
| **ConvNeXt Tiny** | Local textural features (stroke width, ink density) |
| **Swin Transformer Tiny** | Global structural features (spacing, curvature) |
| **Cross-Attention Fusion** | Bidirectional attention between local & global |
| **Projection Head** | Maps fused features to 256-d embedding |

```
Input (224x224x3)
      |
  +---+---+
  |       |
ConvNeXt  Swin
 Tiny     Tiny
  |       |
768-d    768-d
  |       |
  +--X-A--+   Cross-Attention
     |
  1536-d
     |
  512 -> 256-d
```


In [ ]:
class CrossAttentionFusion(nn.Module):
    '''Bidirectional cross-attention: lets local and global branches
    attend to each other before concatenation.'''

    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
        )

    def forward(self, query, key_value):
        q  = self.norm1(query.unsqueeze(1))       # (B,1,D)
        kv = self.norm1(key_value.unsqueeze(1))   # (B,1,D)
        out, _ = self.attn(q, kv, kv)             # (B,1,D)
        fused = query + out.squeeze(1)            # residual
        fused = fused + self.ffn(self.norm2(fused))
        return fused


class HybridBackbone(nn.Module):
    '''ConvNeXt (local) + Swin Transformer (global) with cross-attention fusion.'''

    def __init__(self, embed_dim=256, drop_rate=0.3):
        super().__init__()

        # Branch A: ConvNeXt Tiny (local features)
        self.convnext = timm.create_model(
            'convnext_tiny', pretrained=True, num_classes=0)
        cnx_dim = self.convnext.num_features  # 768

        # Branch B: Swin Transformer Tiny (global features)
        self.swin = timm.create_model(
            'swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)
        swin_dim = self.swin.num_features     # 768

        # Project to common dim
        common_dim = min(cnx_dim, swin_dim)   # 768
        self.proj_local  = nn.Linear(cnx_dim,  common_dim)
        self.proj_global = nn.Linear(swin_dim, common_dim)

        # Cross-Attention Fusion
        self.xattn_l2g = CrossAttentionFusion(common_dim)  # local attends global
        self.xattn_g2l = CrossAttentionFusion(common_dim)  # global attends local

        # Projection Head
        self.head = nn.Sequential(
            nn.Linear(common_dim * 2, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(drop_rate),
            nn.Linear(512, embed_dim),
        )

        self._freeze_early_layers()

    def _freeze_early_layers(self):
        '''Freeze first ~50% of each backbone for transfer learning.'''
        for name, p in self.convnext.named_parameters():
            if any(k in name for k in ['stem', 'stages.0', 'stages.1']):
                p.requires_grad = False
        for name, p in self.swin.named_parameters():
            if any(k in name for k in ['patch_embed', 'layers.0', 'layers.1']):
                p.requires_grad = False

    def forward(self, x):
        f_local  = self.proj_local(self.convnext(x))    # (B, D)
        f_global = self.proj_global(self.swin(x))        # (B, D)

        fl = self.xattn_l2g(f_local,  f_global)  # local enriched by global
        fg = self.xattn_g2l(f_global, f_local)   # global enriched by local

        fused = torch.cat([fl, fg], dim=1)        # (B, 2D)
        return self.head(fused)                   # (B, embed_dim)


class SiameseNetwork(nn.Module):
    '''Twin-branch Siamese wrapper sharing a single backbone.'''

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x1, x2):
        return self.backbone(x1), self.backbone(x2)


### 7.1 Model Summary


In [ ]:
backbone = HybridBackbone(embed_dim=EMBED_DIM)
model    = SiameseNetwork(backbone).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Architecture : Siamese x (ConvNeXt Tiny + Swin Tiny)')
print(f'Total params     : {total:>12,}')
print(f'Trainable params : {trainable:>12,}')
print(f'Frozen params    : {total - trainable:>12,}')
print(f'Embedding dim    : {EMBED_DIM}')


## 8. Contrastive Loss

- y=1 (genuine) -> pull embeddings close (distance -> 0)
- y=0 (forged)  -> push embeddings apart (distance >= margin)


In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=2.0):
        super().__init__()
        self.margin = margin

    def forward(self, e1, e2, label):
        d = F.pairwise_distance(e1, e2, keepdim=True)
        loss = torch.mean(
            label       * d.pow(2) +
            (1 - label) * torch.clamp(self.margin - d, min=0).pow(2)
        )
        return loss


## 9. Training

Features:
- Mixed-precision (FP16) for ~2x speedup
- Gradient clipping (max_norm=1.0)
- CosineAnnealingWarmRestarts LR schedule
- Best-model checkpointing on validation loss


In [ ]:
criterion = ContrastiveLoss(margin=CONTRASTIVE_MARGIN)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=5, T_mult=2)
scaler = GradScaler()

history = {'train_loss': [], 'val_loss': [], 'lr': []}


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running = 0.0
    for img1, img2, labels in loader:
        img1   = img1.to(device, non_blocking=True)
        img2   = img2.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            e1, e2 = model(img1, img2)
            loss = criterion(e1, e2, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running += loss.item() * img1.size(0)

    return running / len(loader.dataset)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running = 0.0
    for img1, img2, labels in loader:
        img1   = img1.to(device, non_blocking=True)
        img2   = img2.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1)
        with autocast():
            e1, e2 = model(img1, img2)
            loss = criterion(e1, e2, labels)
        running += loss.item() * img1.size(0)
    return running / len(loader.dataset)


In [ ]:
best_val_loss = float('inf')
best_weights  = copy.deepcopy(model.state_dict())

print('=' * 65)
print(f'{"Epoch":>6} | {"Train Loss":>11} | {"Val Loss":>11} | '
      f'{"LR":>10} | {"Time":>6}')
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, criterion,
                                  optimizer, scaler, DEVICE)
    val_loss   = validate(model, test_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(optimizer.param_groups[0]['lr'])

    tag = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights  = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(),
                   os.path.join(CHECKPOINT_DIR, 'best_siamese_hybrid.pth'))
        tag = ' *best*'

    elapsed = time.time() - t0
    print(f'{epoch:>6} | {train_loss:>11.4f} | {val_loss:>11.4f} | '
          f'{optimizer.param_groups[0]["lr"]:>10.2e} | {elapsed:>5.1f}s{tag}')

print('=' * 65)
print(f'\nBest validation loss: {best_val_loss:.4f}')

model.load_state_dict(best_weights)
print('Best weights restored.')


### 9.1 Training Curves


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history['train_loss'])+1)

ax1.plot(ep, history['train_loss'], 'b-o', ms=3, label='Train')
ax1.plot(ep, history['val_loss'],   'r-o', ms=3, label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Contrastive Loss'); ax1.legend(); ax1.grid(True)

ax2.plot(ep, history['lr'], 'g-o', ms=3)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('LR')
ax2.set_title('Learning Rate Schedule'); ax2.grid(True)

plt.tight_layout(); plt.show()


## 10. Evaluation

Metrics: ROC-AUC, Equal Error Rate (EER), confusion matrix, classification report.


In [ ]:
@torch.no_grad()
def compute_distances(model, loader, device):
    '''Compute pairwise distances for all pairs.'''
    model.eval()
    dists_all, labels_all = [], []
    for img1, img2, labels in loader:
        img1 = img1.to(device, non_blocking=True)
        img2 = img2.to(device, non_blocking=True)
        with autocast():
            e1, e2 = model(img1, img2)
        d = F.pairwise_distance(e1, e2).cpu().numpy()
        dists_all.extend(d)
        labels_all.extend(labels.numpy())
    return np.array(dists_all), np.array(labels_all)


distances, labels = compute_distances(model, test_loader, DEVICE)
scores = -distances  # negate: higher = more similar

fpr, tpr, thresholds = roc_curve(labels, scores)
roc_auc = auc(fpr, tpr)

fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fnr - fpr))
eer = fpr[eer_idx]
optimal_threshold = -thresholds[eer_idx]

print(f'ROC-AUC              : {roc_auc:.4f}')
print(f'Equal Error Rate     : {eer:.4f}  ({eer*100:.2f}%)')
print(f'Optimal Dist Thresh  : {optimal_threshold:.4f}')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
ax1.plot([0,1],[0,1], 'k--', lw=1, alpha=0.4)
ax1.scatter(fpr[eer_idx], tpr[eer_idx], c='blue', s=100, zorder=5,
            label=f'EER = {eer:.3f}')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curve'); ax1.legend(loc='lower right'); ax1.grid(True)

preds = (distances < optimal_threshold).astype(int)
cm = confusion_matrix(labels.astype(int), preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Forged','Genuine'],
            yticklabels=['Forged','Genuine'], ax=ax2)
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Actual')
ax2.set_title(f'Confusion Matrix (thr={optimal_threshold:.3f})')

plt.tight_layout(); plt.show()

print('\n' + classification_report(
    labels.astype(int), preds,
    target_names=['Forged', 'Genuine']))


## 11. Distance Distribution


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(distances[labels==1], bins=50, alpha=0.7, color='#2ecc71',
        label='Genuine', density=True)
ax.hist(distances[labels==0], bins=50, alpha=0.7, color='#e74c3c',
        label='Forged', density=True)
ax.axvline(optimal_threshold, color='blue', ls='--', lw=2,
           label=f'Threshold = {optimal_threshold:.3f}')
ax.set_xlabel('Euclidean Distance'); ax.set_ylabel('Density')
ax.set_title('Genuine vs Forged Distance Distribution')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 12. t-SNE Embedding Visualisation

Project 256-d embeddings to 2D to inspect per-writer clustering.


In [ ]:
@torch.no_grad()
def extract_embeddings(model, data_dir, writer_ids, transform,
                       device, max_per_writer=10):
    model.eval()
    embs, wids, types = [], [], []
    org_dir  = os.path.join(data_dir, 'full_org')
    forg_dir = os.path.join(data_dir, 'full_forg')

    for wid in writer_ids:
        for fp in sorted(glob.glob(
                os.path.join(org_dir, f'original_{wid}_*.png')))[:max_per_writer]:
            img = transform(Image.open(fp).convert('RGB')).unsqueeze(0).to(device)
            with autocast():
                e = model.backbone(img)
            embs.append(e.cpu().numpy().flatten())
            wids.append(wid); types.append('Genuine')

        for fp in sorted(glob.glob(
                os.path.join(forg_dir, f'forgeries_{wid}_*.png')))[:max_per_writer]:
            img = transform(Image.open(fp).convert('RGB')).unsqueeze(0).to(device)
            with autocast():
                e = model.backbone(img)
            embs.append(e.cpu().numpy().flatten())
            wids.append(wid); types.append('Forged')

    return np.array(embs), np.array(wids), np.array(types)


vis_writers = test_writers[:6]
embs, wlabels, sig_types = extract_embeddings(
    model, DATA_DIR, vis_writers, val_transform, DEVICE)

tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
pts = tsne.fit_transform(embs)

fig, ax = plt.subplots(figsize=(12, 8))
pal = sns.color_palette('husl', len(vis_writers))
for i, wid in enumerate(vis_writers):
    mg = (wlabels == wid) & (sig_types == 'Genuine')
    mf = (wlabels == wid) & (sig_types == 'Forged')
    ax.scatter(pts[mg,0], pts[mg,1], c=[pal[i]], marker='o', s=80,
               edgecolors='k', label=f'W{wid} Genuine', alpha=0.85)
    ax.scatter(pts[mf,0], pts[mf,1], c=[pal[i]], marker='x', s=80,
               label=f'W{wid} Forged', alpha=0.6)

ax.set_title('t-SNE  -  256-d Embeddings', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05,1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 13. Single-Pair Inference

Drop in any two signature images and get a verdict.


In [ ]:
def verify_signatures(model, path1, path2, threshold, transform, device):
    '''Returns (is_genuine, distance, confidence).'''
    model.eval()
    img1 = transform(Image.open(path1).convert('RGB')).unsqueeze(0).to(device)
    img2 = transform(Image.open(path2).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad(), autocast():
        e1, e2 = model(img1, img2)
    dist = F.pairwise_distance(e1, e2).item()
    is_genuine = dist < threshold
    confidence = torch.sigmoid(torch.tensor(threshold - dist)).item()
    return is_genuine, dist, confidence


# Demo
demo_org  = sorted(glob.glob(
    os.path.join(DATA_DIR, 'full_org', f'original_{test_writers[0]}_1.png')))
demo_forg = sorted(glob.glob(
    os.path.join(DATA_DIR, 'full_forg', f'forgeries_{test_writers[0]}_1.png')))

if demo_org and demo_forg:
    ok, d, c = verify_signatures(
        model, demo_org[0], demo_forg[0],
        optimal_threshold, val_transform, DEVICE)

    verdict = 'GENUINE' if ok else 'FORGED'
    vcolor  = '#2ecc71' if ok else '#e74c3c'
    print(f'A : {os.path.basename(demo_org[0])}')
    print(f'B : {os.path.basename(demo_forg[0])}')
    print(f'Dist      : {d:.4f}   |  Threshold : {optimal_threshold:.4f}')
    print(f'Verdict   : {verdict}  (confidence {c*100:.1f}%)')

    fig, (a1,a2) = plt.subplots(1,2,figsize=(8,4))
    a1.imshow(Image.open(demo_org[0]));  a1.set_title('Image A'); a1.axis('off')
    a2.imshow(Image.open(demo_forg[0])); a2.set_title('Image B'); a2.axis('off')
    fig.suptitle(f'{verdict} -- dist={d:.3f}, conf={c*100:.1f}%',
                 fontsize=14, fontweight='bold', color=vcolor)
    plt.tight_layout(); plt.show()
else:
    print('Demo images not found - adjust paths above.')


## 14. Model Export


In [ ]:
# Full checkpoint
ckpt = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'threshold': optimal_threshold,
    'eer': eer,
    'roc_auc': roc_auc,
    'embed_dim': EMBED_DIM,
    'hyperparameters': {
        'img_size': IMG_SIZE, 'margin': CONTRASTIVE_MARGIN,
        'lr': LEARNING_RATE, 'epochs': NUM_EPOCHS,
    }
}
ckpt_path = os.path.join(CHECKPOINT_DIR, 'siamese_hybrid_checkpoint.pth')
torch.save(ckpt, ckpt_path)
print(f'Checkpoint saved : {ckpt_path}')

# TorchScript export
model.eval()
dummy = (torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE),
         torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE))
try:
    traced = torch.jit.trace(model, dummy)
    ts_path = os.path.join(CHECKPOINT_DIR, 'siamese_hybrid_traced.pt')
    traced.save(ts_path)
    print(f'TorchScript saved: {ts_path}')
except Exception as e:
    print(f'TorchScript export failed: {e}')
    print('Use the .pth checkpoint for inference instead.')


## 15. Summary

| Component | Detail |
|-----------|--------|
| **Architecture** | Siamese x (ConvNeXt Tiny + Swin Tiny) |
| **Fusion** | Cross-Attention Module (bidirectional) |
| **Loss** | Contrastive (margin = 2.0) |
| **Embedding** | 256-d |
| **Optimiser** | AdamW + CosineAnnealingWarmRestarts |
| **Training** | Mixed-precision FP16, gradient clipping |
| **Dataset** | CEDAR (55 writers, 24 samples each) |

### Next Steps
- Experiment with **triplet loss** or **ArcFace loss**
- Add **hard negative mining**
- Fine-tune `MARGIN`, `EMBED_DIM`, augmentation intensity
- Deploy via **ONNX** or **TorchServe**
